# 1.2 后端接口与执行模型

## 本节目标

- 理解同一 benchmark 如何替换多个 SpMV backend
- 用构建依赖和数据类型判断代码实际运行位置

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
import platform, shutil
print("Python:", platform.python_version())
print("CMake:", shutil.which("cmake"))
print("正式路径：Ascend C FP32 RTC；FP16/BF16/persistent 仍为 Host Prototype")


## 从 README 目标落到代码

原工程要比较 CPU single、OpenMP16 和多种“优化后端”。共同接口是 `ISpmvBackend`：`prepare()` 接收不变的 CSR，`run()` 接收每轮变化的 `x` 并产生 `y`。这种分离为 persistent matrix 和 cold/warm 计时提供了结构基础。

判断实现边界不能只看类名。当前 CMake 默认 `SPMV_REAL_ASCENDC=ON`：查找 `acl/acl.h`、`ascendcl`、`acl_rtc`，编译 `ascendc_spmv.cpp`（真实类 `AscendCSpmvBackend`）并链接 Device 库；`npu_spmv*.cpp` 与 `npu_spmv_context.cpp` 仍是 Host 主机循环（Host Prototype）。

## 调用链

`spmv_benchmark.cpp` 对每个 backend 先调用一次 `prepare()`，再执行 warmup 和 repeat。它把每次 `run()` 返回的 transfer/kernel/total 字段平均，并统一与 CPU reference 计算误差。`host_prototype_*` 字段是主机原型阶段划分，不自动证明发生了 Host/Device 传输或 Device Kernel；只有 `ascendc_*` 字段（如 `ascendc_transfer_in_ms`、`ascendc_launch_to_complete_ms`、`ascendc_transfer_out_ms`）来自 `AscendCSpmvBackend` 的真实 `aclrtMemcpy` 与 RTC 启动，对应日志 `Actual Backend=Ascend C RTC`。

## 观察与解释

接口设计是可迁移的：benchmark 不感知 backend 内部实现。当前 FP32 正式路径已替换为真实 `AscendCSpmvBackend`（RTC 编译/加载/启动），Host 原型仍用于验证算法、精度和生命周期选择。

## 课后实践

列出判断一个 backend 是否真实运行在 Ascend Device 上所需的三类证据，并说明 FP32 正式路径（`AscendCSpmvBackend`）与 FP16/BF16/persistent Host 原型各自具备/缺少哪些。参考答案见 `answer/01.02_answer.md`。